# TabICLv2 NBA Evaluation — Colab T4

**Goal**: Test TabICLv2 (MIT, soda-inria) against our current tree models.
Expected: -0.005 to -0.008 Brier improvement.

**Instructions**: Runtime → Change runtime type → GPU (T4) → Run All

In [ ]:
import subprocess, sys, os

# Clone repo
if not os.path.exists('/content/nomos-nba-agent'):
    subprocess.run(['git', 'clone', 'https://github.com/LBJLincoln/nomos-nba-agent.git',
                    '/content/nomos-nba-agent'], check=True)
else:
    subprocess.run(['git', '-C', '/content/nomos-nba-agent', 'pull'], check=True)

os.chdir('/content/nomos-nba-agent')
sys.path.insert(0, '/content/nomos-nba-agent')

# Install deps
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'tabicl', 'tabpfn', 'xgboost', 'lightgbm', 'catboost',
    'scikit-learn>=1.3', 'nba_api', 'pandas', 'numpy', 'scipy', 'mapie'])

import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 1. Build Features (NBAFeatureEngine v3.0)

In [ ]:
import json, numpy as np, time as _time
from pathlib import Path

CACHE_FILE = Path('/content/features_cache.npz')

if CACHE_FILE.exists():
    print('Loading cached feature matrix...')
    _t0 = _time.time()
    cached = np.load(CACHE_FILE, allow_pickle=True)
    X = cached['X']
    y = cached['y']
    feature_names = cached['feature_names'].tolist()
    print(f'Loaded from cache in {_time.time()-_t0:.1f}s: {X.shape} ({len(feature_names)} features)')
else:
    print('No cache found — building features (this takes ~30 min, will cache for next run)...')
    # Load cached game data
    data_dir = Path('/content/nomos-nba-agent/hf-space/data/historical')
    games = []
    for f in sorted(data_dir.glob('games-*.json')):
        raw = json.loads(f.read_text())
        if isinstance(raw, list):
            games.extend(raw)
        elif isinstance(raw, dict) and 'games' in raw:
            games.extend(raw['games'])
        else:
            print(f'WARNING: unexpected format in {f.name}, skipping')
    print(f'Games loaded: {len(games)}')

    # Build features
    import sys
    sys.path.insert(0, '/content/nomos-nba-agent/hf-space')
    from features.engine import NBAFeatureEngine
    _t0 = _time.time()
    engine = NBAFeatureEngine()
    X, y, feature_names = engine.build(games)
    X = np.nan_to_num(np.array(X, dtype=np.float64))
    y = np.array(y, dtype=np.int32)
    print(f'Built in {_time.time()-_t0:.0f}s')

    # Remove zero-variance features
    variances = np.var(X, axis=0)
    valid = variances > 1e-10
    X = X[:, valid]
    feature_names = [f for f, v in zip(feature_names, valid) if v]

    # Save cache for next runs
    np.savez_compressed(CACHE_FILE, X=X, y=y, feature_names=np.array(feature_names))
    print(f'Cached to {CACHE_FILE} ({CACHE_FILE.stat().st_size / 1e6:.1f} MB)')

print(f'Feature matrix: {X.shape} ({len(feature_names)} usable features)')

# Pre-compute variance for later use
variances = np.var(X, axis=0)

## 2. Get S10 Best Feature Subset

In [ ]:
import requests

# Fetch S10's evolved best feature set
S10_URL = 'https://lbjlincoln-nomos-nba-quant.hf.space'
try:
    resp = requests.get(f'{S10_URL}/api/results', timeout=30)
    best = resp.json()['best']
    s10_features = best['selected_features']
    print(f'S10 best: brier={best["brier"]}, model={best["model_type"]}, features={best["n_features"]}')
    print(f'Feature names: {s10_features[:10]}...')
    
    # Map S10 feature names to indices in our matrix
    name_to_idx = {name: i for i, name in enumerate(feature_names)}
    s10_indices = [name_to_idx[f] for f in s10_features if f in name_to_idx]
    print(f'Mapped {len(s10_indices)}/{len(s10_features)} features to our matrix')
    X_s10 = X[:, s10_indices]
    print(f'S10 feature matrix: {X_s10.shape}')
except Exception as e:
    print(f'Could not fetch S10 features: {e}')
    print('Using top 100 features by variance instead')
    top_idx = np.argsort(variances[valid])[-100:]
    X_s10 = X[:, top_idx]
    s10_indices = top_idx.tolist()

## 3. Walk-Forward Comparison: TabICLv2 vs Tree Models

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.base import clone
import xgboost as xgb
import time

# Use S10's evolved feature subset
X_eval = np.nan_to_num(X_s10, nan=0.0, posinf=1e6, neginf=-1e6)
n_splits = 5
PURGE_GAP = 5

# Use GPU for XGBoost if available
_xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'XGBoost device: {_xgb_device}')

models = {
    'random_forest': lambda: RandomForestClassifier(n_estimators=200, max_depth=8,
        min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1),
    'extra_trees': lambda: ExtraTreesClassifier(n_estimators=200, max_depth=8,
        min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1),
    'xgboost': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=8,
        learning_rate=0.02, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.01, reg_lambda=0.1, random_state=42,
        tree_method='hist', device=_xgb_device),
}

# Add TabICLv2
try:
    from tabicl import TabICLClassifier
    models['tabicl_v2'] = lambda: TabICLClassifier()
    print('TabICLv2: loaded OK')
except ImportError as e:
    print(f'TabICLv2 import failed: {e}')

# Add TabPFN-2.5
try:
    from tabpfn import TabPFNClassifier
    models['tabpfn_2.5'] = lambda: TabPFNClassifier(device='cuda' if torch.cuda.is_available() else 'cpu')
    print('TabPFN-2.5: loaded OK')
except ImportError as e:
    print(f'TabPFN-2.5 import failed: {e}')

# Walk-forward evaluation
tscv = TimeSeriesSplit(n_splits=n_splits)
results = {name: [] for name in models}
timings = {name: [] for name in models}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_eval)):
    # Purge gap to prevent temporal leakage
    train_idx = train_idx[:-PURGE_GAP] if len(train_idx) > PURGE_GAP + 50 else train_idx
    X_train, X_test = X_eval[train_idx], X_eval[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print(f'\nFold {fold+1}/{n_splits}: train={len(train_idx)}, test={len(test_idx)}')
    
    for name, model_fn in models.items():
        try:
            t0 = time.time()
            model = model_fn()
            model.fit(X_train, y_train)
            probs = model.predict_proba(X_test)[:, 1]
            elapsed = time.time() - t0
            brier = brier_score_loss(y_test, probs)
            results[name].append(brier)
            timings[name].append(elapsed)
            print(f'  {name:20s}: brier={brier:.5f} ({elapsed:.1f}s)')
        except Exception as e:
            print(f'  {name:20s}: FAILED ({e})')
            results[name].append(0.30)
            timings[name].append(0)

print('\n' + '='*70)
print('FINAL RESULTS (walk-forward, lower = better)')
print('='*70)
for name, briers in sorted(results.items(), key=lambda x: np.mean(x[1])):
    mean_b = np.mean(briers)
    std_b = np.std(briers)
    mean_t = np.mean(timings[name])
    print(f'  {name:20s}: {mean_b:.5f} +/- {std_b:.5f}  ({mean_t:.1f}s/fold)')

best_tree = min(np.mean(results[n]) for n in ['random_forest', 'extra_trees', 'xgboost'] if n in results)
if 'tabicl_v2' in results:
    delta = np.mean(results['tabicl_v2']) - best_tree
    print(f'\nTabICLv2 vs best tree: {delta:+.5f} ({"BETTER" if delta < 0 else "WORSE"})')
if 'tabpfn_2.5' in results:
    delta = np.mean(results['tabpfn_2.5']) - best_tree
    print(f'TabPFN-2.5 vs best tree: {delta:+.5f} ({"BETTER" if delta < 0 else "WORSE"})')

## 4. Stacking: TabICLv2 as Meta-Learner

In [ ]:
# Stack: use tree model predictions as input to TabICLv2 meta-learner
from sklearn.base import clone as sk_clone

base_models = {
    'rf': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1),
    'et': ExtraTreesClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1),
    'xgb': xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.02, random_state=42,
        tree_method='hist', device=_xgb_device),
}

stack_briers = []
base_briers = {n: [] for n in base_models}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_eval)):
    train_idx = train_idx[:-PURGE_GAP] if len(train_idx) > PURGE_GAP + 50 else train_idx
    X_train, X_test = X_eval[train_idx], X_eval[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 80/20 split for base-train / meta-train
    split_pt = int(len(train_idx) * 0.8)
    base_tr, meta_tr = train_idx[:split_pt], train_idx[split_pt:]
    
    meta_train_feat, meta_test_feat = [], []
    for name, model in base_models.items():
        m = sk_clone(model)
        m.fit(X_eval[base_tr], y[base_tr])
        meta_train_feat.append(m.predict_proba(X_eval[meta_tr])[:, 1])
        meta_test_feat.append(m.predict_proba(X_test)[:, 1])
        base_briers[name].append(brier_score_loss(y_test, m.predict_proba(X_test)[:, 1]))
    
    X_meta_train = np.column_stack(meta_train_feat)
    X_meta_test = np.column_stack(meta_test_feat)
    
    try:
        meta = TabICLClassifier()
        meta.fit(X_meta_train, y[meta_tr])
        stack_probs = meta.predict_proba(X_meta_test)[:, 1]
        sb = brier_score_loss(y_test, stack_probs)
        stack_briers.append(sb)
        print(f'Fold {fold+1}: stack={sb:.5f} | rf={base_briers["rf"][-1]:.5f} et={base_briers["et"][-1]:.5f} xgb={base_briers["xgb"][-1]:.5f}')
    except Exception as e:
        print(f'Fold {fold+1}: stack FAILED ({e})')
        stack_briers.append(0.30)

print('\n' + '='*70)
print('STACKING RESULTS')
print('='*70)
print(f'  TabICLv2 stacking: {np.mean(stack_briers):.5f}')
for name, briers in base_briers.items():
    print(f'  {name} alone:       {np.mean(briers):.5f}')
best_base = min(np.mean(v) for v in base_briers.values())
delta = np.mean(stack_briers) - best_base
print(f'\n  Stacking delta: {delta:+.5f} ({"BETTER" if delta < 0 else "WORSE"})')

## 5. Rank-Based Ensemble Fusion Test

In [ ]:
# Test rank-based fusion (arXiv 2603.10916) vs simple averaging
avg_briers, rank_briers = [], []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_eval)):
    train_idx = train_idx[:-PURGE_GAP] if len(train_idx) > PURGE_GAP + 50 else train_idx
    X_train, X_test = X_eval[train_idx], X_eval[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    all_probs = {}
    for name, model_fn in [('rf', base_models['rf']), ('et', base_models['et']), ('xgb', base_models['xgb'])]:
        m = sk_clone(model_fn)
        m.fit(X_train, y_train)
        all_probs[name] = m.predict_proba(X_test)[:, 1]
    
    n_test = len(test_idx)
    # Simple average
    avg_probs = np.mean([all_probs[n] for n in all_probs], axis=0)
    avg_briers.append(brier_score_loss(y_test, avg_probs))
    
    # Rank-based fusion
    rank_probs = np.zeros(n_test)
    for i in range(n_test):
        probs_i = [(n, all_probs[n][i]) for n in all_probs]
        sorted_p = sorted(probs_i, key=lambda x: x[1])
        ranks = {n: (j+1)/len(sorted_p) for j, (n, _) in enumerate(sorted_p)}
        avg_rank = np.mean(list(ranks.values()))
        min_p = min(p for _, p in probs_i)
        max_p = max(p for _, p in probs_i)
        rank_probs[i] = min_p + avg_rank * (max_p - min_p) if max_p > min_p else np.mean([p for _, p in probs_i])
    rank_briers.append(brier_score_loss(y_test, rank_probs))
    
    print(f'Fold {fold+1}: avg={avg_briers[-1]:.5f} rank={rank_briers[-1]:.5f}')

print(f'\nSimple average: {np.mean(avg_briers):.5f}')
print(f'Rank fusion:    {np.mean(rank_briers):.5f}')
print(f'Delta:          {np.mean(rank_briers) - np.mean(avg_briers):+.5f}')